# Indexation des documents FinanceBench avec BGE-M3

Ce notebook calcule les vecteurs des documents avec **BGE-M3** (sentence-transformers) sur la carte graphique gratuite de Colab (229 documents : tous les 10-Q, 8-K et communiqués de résultats, les 10-K depuis 2020 et tous les documents qui ont des questions FinanceBench), puis les envoie dans **Qdrant Cloud**.

**Avant de commencer :**
1. Menu **Exécution → Modifier le type d'exécution → T4 GPU → Enregistrer**.
2. Icône **clé** à gauche (**Secrets**) : ajoutez `QDRANT_URL` et `QDRANT_API_KEY` (et `GITHUB_TOKEN` si votre dépôt est privé), puis activez l'accès au notebook.

Lancez ensuite les cellules **une par une**, de haut en bas.

## 1. Vérifier que la carte graphique est active
Vous devez voir un tableau avec **Tesla T4**. Si vous voyez une erreur, refaites l'étape 1 ci-dessus.

In [ ]:
!nvidia-smi

## 2. Récupérer le code du projet depuis GitHub
Remplacez `VOTRE_COMPTE` par votre nom d'utilisateur GitHub.

In [ ]:
import os
from google.colab import userdata

REPO = "VOTRE_COMPTE/financial-analyst"

try:
    token = userdata.get("GITHUB_TOKEN")  # seulement si le dépôt est privé
    url = f"https://{token}@github.com/{REPO}.git"
except Exception:
    url = f"https://github.com/{REPO}.git"

# Chemins absolus : on peut relancer cette cellule sans créer de dossiers imbriqués
%cd /content
!rm -rf /content/financial-analyst
!git clone -q {url} /content/financial-analyst
%cd /content/financial-analyst
!ls src

## 3. Installer les bibliothèques
PyTorch est déjà installé sur Colab. La cellule affiche ensuite les versions de `langchain-huggingface` et `sentence-transformers` : si l'une manque, votre `requirements.txt` sur GitHub n'est pas à jour.

In [ ]:
!pip install -q -r requirements.txt
# Vérification : ces deux lignes doivent afficher un numéro de version, sans erreur
!pip show langchain-huggingface | grep Version
!pip show sentence-transformers | grep Version

## 4. Charger les clés Qdrant depuis les Secrets

In [ ]:
os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
print("Clés chargées.")

## Test 1 : connexion à Qdrant
Doit afficher « Connexion à Qdrant : OK ».

In [ ]:
!python src/test_connexion.py

## Test 2 : le modèle BGE-M3
Télécharge le modèle (quelques minutes la première fois). La similarité anglais ↔ français doit être nettement plus élevée que anglais ↔ sans rapport.

In [ ]:
!python src/test_embeddings.py

## 5. Lancer l'indexation (version accélérée, 229 documents)
Les documents déjà dans Qdrant sont sautés. Chaque ligne affiche le **temps restant estimé**.

Gardez cet onglet ouvert. Si Colab se déconnecte, relancez les cellules 2 à 5 : l'indexation reprend là où elle s'était arrêtée.

In [ ]:
!python src/indexing.py --tout

## Test 3 : tous les documents sont-ils dans la base ?
Doit afficher « Documents manquants : 0 ».

In [ ]:
!python src/verifier_base.py

## Test 4 : la recherche trouve-t-elle les bons passages ?
D'abord sur une question en anglais et en français :

In [ ]:
!python src/test_recherche.py

Puis sur les 150 questions de FinanceBench (quelques minutes). Notez les pourcentages : ce sont vos premiers résultats d'évaluation.

In [ ]:
!python evaluation/test_retrieval.py